In [1]:
# Load preprocessed and augmented data from 01_preprocessing.ipynb
%run 01_preprocessing.ipynb

Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.



Data augmentation complete!
Original rows (body only): 2029
Augmented rows (body + examples): 8310
New rows added: 6281

Label distribution:
rule_violation
1    4229
0    4081
Name: count, dtype: int64

Columns in new df (examples removed): ['body', 'rule', 'subreddit', 'rule_violation', 'source']

Applied combine_comment_rule to augmented dataset
Final columns after augmentation and unique comment-rule pair combination: ['body', 'rule', 'subreddit', 'rule_violation', 'source', 'combined_text']
Total rows after cleaning: 8310


In [2]:
from preprocessing import write_positive_negative_label, split_data, map_fasttext_labels
from models.fasttext import write_fasttext_file, fasttext_predict
from eval import evaluate_model
import fasttext

In [3]:
train_df_clean = write_positive_negative_label(train_df_clean)

In [4]:
train_df_clean.head()

,body,rule,subreddit,rule_violation,source,combined_text,label
0,FindSexToday . com - free girls for sex worldw...,"No Advertising: Spam, referral links, unsolici...",gonewild,1,train_aug_positive,FindSexToday . com - free girls for sex worldw...,positive
1,see there hooters sweet <URL>,"No Advertising: Spam, referral links, unsolici...",BlackPeopleTwitter,1,train_aug_positive,see there hooters sweet <URL> [SEP] No Adverti...,positive
2,buy my shirt because i want to buy ganja!: <URL>,"No Advertising: Spam, referral links, unsolici...",soccerstreams,1,train_aug_positive,buy my shirt because i want to buy ganja!: <UR...,positive
3,Smashing evidence under subpoena is not violat...,No legal advice: Do not offer or request legal...,politics,1,original,Smashing evidence under subpoena is not violat...,positive
4,"if she doesn't want to be a ""right wing target...",No legal advice: Do not offer or request legal...,hillaryclinton,1,original,"if she doesn't want to be a ""right wing target...",positive


In [4]:
# Train and validation split (on fasttext labelled data)
train_df, val_df = split_data(train_df_clean, label_column='rule_violation')

In [6]:
write_fasttext_file(
    df=train_df,
    text_col="combined_text",
    label_col="rule_violation",
    out_path="../data/fasttext/fasttext_train.txt"
)
write_fasttext_file(
    df=val_df,
    text_col="combined_text",
    label_col="rule_violation",
    out_path="../data/fasttext/fasttext_val.txt"
)

In [11]:
for i in range(5):
    print(repr(open("../data/fasttext/fasttext_train.txt", "r").readline()))

"__label__1 You don't have to give a boyfriend notice if he isn't on the lease; you can kick him out any time he makes you feel unsafe or creeped out. Refund him his share of the rent, pro-rated. When he is out of the house, remove his things and have that supportive landlord get the locks changed. You don't have to put up with this. He can take you to court if he wants to; you don't have to live in a situation that makes you feel like you aren't safe. [SEP] No legal advice: Do not offer or request legal advice.\n"
"__label__1 You don't have to give a boyfriend notice if he isn't on the lease; you can kick him out any time he makes you feel unsafe or creeped out. Refund him his share of the rent, pro-rated. When he is out of the house, remove his things and have that supportive landlord get the locks changed. You don't have to put up with this. He can take you to court if he wants to; you don't have to live in a situation that makes you feel like you aren't safe. [SEP] No legal advice:

In [7]:
model = fasttext.train_supervised(
    input="../data/fasttext/fasttext_train.txt",
    lr=0.5,
    epoch=5,
    wordNgrams=2
)

Read 0M words
Number of words:  11030
Number of labels: 2
Progress: 100.0% words/sec/thread: 1496142 lr:  0.000000 avg.loss:  0.176160 ETA:   0h 0m 0s


In [ ]:
model.save_model("fasttext_baseline.bin")

In [ ]:
# Load model
model = fasttext.load_model("fasttext_baseline.bin")
y_pred_labels, y_pred_probs = fasttext_predict(model, X_val.tolist())

In [ ]:
# Evaluate
evaluate_model(y_val, y_pred_labels, y_pred_probs, title="FastText Baseline")